# NumPy 2 – Filtern und Mengen

**Szenario:** Wir bleiben bei der *Kiez Kaffee GmbH* mit ihren vier Filialen.
Heute geht es um **Kundinnen und Kunden**: Wer kauft wo? Wer kommt öfter? Wer kauft in mehreren Filialen?

**Heute:** Mengen in Python und NumPy · Duplikate finden · Schnittmenge, Vereinigung, Differenz · bedingtes Ersetzen mit `np.where` · Fancy Indexing

In [1]:
import numpy as np

# A – Brücke zur Mathe: Mengenlehre

## 1 – Mengen in Python

Heute Vormittag hast du Mengen kennengelernt. Zwei Eigenschaften sind für uns wichtig:

- Eine Menge enthält jedes Element **nur einmal** (keine Duplikate).
- Die **Reihenfolge** spielt keine Rolle.

Python hat dafür einen eigenen Datentyp: `set`. Die Operatoren kennst du schon aus der Booleschen Algebra:

| Mathe | Bedeutung | Python-`set` |
|---|---|---|
| A ∩ B | Schnittmenge: in A **und** in B | `A & B` |
| A ∪ B | Vereinigung: in A **oder** in B | `A \| B` |
| A \ B | Differenz: in A, aber **nicht** in B | `A - B` |
| A △ B | symmetrische Differenz: in **genau einer** der beiden | `A ^ B` |
| x ∈ A | x ist Element von A | `x in A` |
| \|A\| | Mächtigkeit (Anzahl Elemente) | `len(A)` |

Das ist kein Zufall: **UND** entspricht der Schnittmenge, **ODER** der Vereinigung, **XOR** der symmetrischen Differenz.

In [ ]:
# Kunden-IDs/ Nummer, die in dieser Woche in Kreuzberg und in Neukölln gekauft haben
kb = {101, 102, 103, 104}   # Kreuzberg
nk = {103, 104, 105, 106}   # Neukölln

print("Schnittmenge      :", kb & nk)
print("Vereinigung       :", kb | nk)
print("nur Kreuzberg     :", kb - nk)
print("genau eine Filiale:", kb ^ nk)
print("101 in Kreuzberg? :", 101 in kb)
print("Anzahl Kreuzberg  :", len(kb))

Schnittmenge      : {104, 103}
Vereinigung       : {101, 102, 103, 104, 105, 106}
nur Kreuzberg     : {101, 102}
genau eine Filiale: {101, 102, 105, 106}
101 in Kreuzberg? : True
Anzahl Kreuzberg  : 4


## 2 – Warum dann NumPy?

In echten Daten stehen Kunden-IDs meist in einer **Liste von Bestellungen**. Wer dreimal kauft, steht dreimal drin.
Solche Daten liegen als **Array** vor, und wir wollen damit weiterrechnen, filtern und zählen.

NumPy hat deshalb eigene Mengen-Funktionen, die direkt mit Arrays arbeiten. Die lernst du in den Blöcken B und C kennen.
Der Unterschied zu `set`: Die Ergebnisse sind wieder **Arrays** und immer **sortiert**.

### Übung 1 – Mengen mit set

👥 **Gemeinsame Übung**

Die Filiale Friedrichshain hatte diese Woche die Kunden `{104, 106, 107, 108}`.

1. Welche Kunden waren in **Kreuzberg und Friedrichshain**?
2. Welche Kunden waren in **mindestens einer** der drei Filialen (Kreuzberg, Neukölln, Friedrichshain)?
3. Welche Kunden waren **nur in Friedrichshain** (in keiner der anderen beiden)?
4. Wie viele verschiedene Kunden gab es insgesamt?

In [ ]:
# Dein Code hier
kb = {101, 102, 103, 104}   # Kreuzberg
nk = {103, 104, 105, 106}   # Neukölln
fh = {104, 106, 107, 108}   # Friedrichshain

# 1
print(kb & fh)
# 2
alle = kb | nk | fh
print(alle)
# 3
print(fh - kb - nk)     # fh - (kb | nk)
# 4
print(len(alle))

{104}
{101, 102, 103, 104, 105, 106, 107, 108}
{107, 108}
8


# B – Duplikate und Zugehörigkeit: `np.unique` und `np.isin`

## 3 – `np.unique`: jedes Element nur einmal

`np.unique(arr)` macht aus einem Array eine **Menge**: Duplikate fallen weg, das Ergebnis ist sortiert.
Mit `len()` bekommst du die **Mächtigkeit**, also die Anzahl verschiedener Werte.

In [4]:
# Kunden-IDs der Bestellungen in Kreuzberg (eine ID pro Bestellung)
bestellungen_kb = np.array([101, 102, 103, 101, 104, 102, 101, 105])

kunden_kb = np.unique(bestellungen_kb)
print("Verschiedene Kunden:", kunden_kb)
print("Anzahl Bestellungen:", len(bestellungen_kb))
print("Anzahl Kunden      :", len(kunden_kb))

Verschiedene Kunden: [101 102 103 104 105]
Anzahl Bestellungen: 8
Anzahl Kunden      : 5


## 4 – Wie oft kommt jeder Wert vor? `return_counts=True`

Mit dem Zusatz `return_counts=True` liefert `np.unique` **zwei** Arrays zurück:
die verschiedenen Werte **und** wie oft jeder vorkommt.

In [38]:
bestellungen_kb = np.array([101, 102, 103, 101, 104, 102, 101, 105])
kunden, anzahl = np.unique(bestellungen_kb, return_counts=True)
print("Kunde :", kunden)
print("Anzahl:", anzahl)

Kunde : [101 102 103 104 105]
Anzahl: [3 2 1 1 1]


Die beiden Arrays gehören **Stelle für Stelle** zusammen. Deshalb kann man mit einer Maske aus `anzahl` direkt `kunden` filtern:

In [6]:
# Kunden mit mindestens 2 Bestellungen
print(kunden[anzahl >= 2])

[101 102]


## 5 – `np.isin`: ist ein Element in der Menge? (∈)

`np.isin(arr, menge)` prüft für **jedes Element** von `arr`, ob es in `menge` vorkommt.
Das Ergebnis ist eine **Maske**, die du wie gewohnt zum Filtern benutzen kannst.

| Mathe | NumPy |
|---|---|
| x ∈ M | `np.isin(arr, M)` |
| x ∉ M | `~np.isin(arr, M)` |

In [ ]:
bestellungen_kb = np.array([101, 102, 103, 101, 104, 102, 101, 105])
stammkunden = np.array([101, 103, 110])

maske = np.isin(bestellungen_kb, stammkunden) # arr=bestellungen_kb, M=stammkunden
print("Maske               :", maske)
print("Bestellungen Stamm  :", bestellungen_kb[maske])
print("Bestellungen andere :", bestellungen_kb[~maske])

Maske               : [ True False  True  True False False  True False]
Bestellungen Stamm  : [101 103 101 101]
Bestellungen andere : [102 104 102 105]


> 💡 **Kurz erwähnt:** `np.isin` funktioniert auch mit Texten, z. B. `np.isin(produkte, ["Espresso", "Crema"])`.

Die Bestellungen in Neukölln:

In [8]:
bestellungen_nk = np.array([103, 104, 105, 105, 106, 104, 105, 107, 103, 105])

### Übung 2 – Kunden in Neukölln

👥 **Gemeinsame Übung**

1. Wie viele **Bestellungen** und wie viele **verschiedene Kunden** hatte Neukölln?
2. Wie oft hat jeder Kunde bestellt?
3. Welcher Kunde hat **am häufigsten** bestellt? (Tipp: `anzahl.max()` und eine Maske)
4. Wie viele Bestellungen kamen von `stammkunden`?

In [ ]:
bestellungen_nk = np.array([103, 104, 105, 105, 106, 104, 105, 107, 103, 105])
stammkunden = np.array([101, 103, 110])
# Dein Code hier
# 1
print(len(bestellungen_nk), len(np.unique(bestellungen_nk)))
# 2
kunde_nk, anzahl_nk = np.unique(bestellungen_nk, return_counts=True)
print(kunde_nk)
print(anzahl_nk)
# 3
print(kunde_nk[anzahl_nk == anzahl_nk.max()])
# 4
print(len(bestellungen_nk[np.isin(bestellungen_nk, stammkunden)]))

10 5
[103 104 105 106 107]
[2 2 4 1 1]
[105]
2


# C – Mengenoperationen in NumPy

## 6 – Schnittmenge, Vereinigung, Differenz

| Mathe | `set` | NumPy |
|---|---|---|
| A ∩ B | `A & B` | `np.intersect1d(A, B)` |
| A ∪ B | `A \| B` | `np.union1d(A, B)` |
| A \ B | `A - B` | `np.setdiff1d(A, B)` |
| A △ B | `A ^ B` | `np.setxor1d(A, B)` |

Alle vier Funktionen entfernen **automatisch Duplikate** und liefern ein **sortiertes Array**.
Die `1d` im Namen heißt: Sie arbeiten mit 1D-Arrays.

In [ ]:
bestellungen_nk = np.array([103, 104, 105, 105, 106, 104, 105, 107, 103, 105])
bestellungen_kb = np.array([101, 102, 103, 101, 104, 102, 101, 105])

print("Beide Filialen   :", np.intersect1d(bestellungen_kb, bestellungen_nk))
print("Mind. eine       :", np.union1d(bestellungen_kb, bestellungen_nk))
print("Nur Kreuzberg    :", np.setdiff1d(bestellungen_kb, bestellungen_nk))
print("Nur Neukölln     :", np.setdiff1d(bestellungen_nk, bestellungen_kb))
print("Genau eine       :", np.setxor1d(bestellungen_kb, bestellungen_nk))
print(bestellungen_nk)

Beide Filialen   : [103 104 105]
Mind. eine       : [101 102 103 104 105 106 107]
Nur Kreuzberg    : [101 102]
Nur Neukölln     : [106 107]
Genau eine       : [101 102 106 107]


**Achtung bei der Differenz:** Die Reihenfolge ist wichtig! A \ B ist etwas anderes als B \ A, genau wie in der Mathe.

## 7 – Mehr als zwei Mengen

Die Funktionen nehmen immer nur **zwei** Arrays. Für drei Mengen schachtelt man sie:

A ∩ B ∩ C → `np.intersect1d(np.intersect1d(A, B), C)`

In [ ]:
bestellungen_nk = np.array([103, 104, 105, 105, 106, 104, 105, 107, 103, 105])
bestellungen_kb = np.array([101, 102, 103, 101, 104, 102, 101, 105])
bestellungen_fh = np.array([104, 106, 107, 108, 104])

in_allen_drei = np.intersect1d(np.intersect1d(bestellungen_kb, bestellungen_nk), bestellungen_fh)
print("In allen drei Filialen:", in_allen_drei)

In allen drei Filialen: [104]


Die Sortimente von Kreuzberg und Wedding:

In [12]:
sortiment_kb = np.array(["Espresso", "Crema", "Filter", "Decaf", "Chai"])
sortiment_we = np.array(["Crema", "Filter", "Kakao", "Chai", "Matcha"])

### Übung 3 – Sortiment vergleichen

👥 **Gemeinsame Übung**

Nutze `sortiment_kb` und `sortiment_we` aus der Zelle oben.

1. Welche Produkte gibt es in **beiden** Filialen?
2. Welche Produkte gibt es **insgesamt**?
3. Welche Produkte gibt es **nur in Kreuzberg**?
4. Welche Produkte gibt es nur in **einer** der beiden Filialen?

In [44]:
sortiment_kb = np.array(["Espresso", "Crema", "Filter", "Decaf", "Chai"])
sortiment_we = np.array(["Crema", "Filter", "Kakao", "Chai", "Matcha"])
# Dein Code hier
# 1
print(np.intersect1d(sortiment_kb, sortiment_we))
# 2
print(np.union1d(sortiment_kb, sortiment_we))
# 3
print(np.setdiff1d(sortiment_kb, sortiment_we))
# 4
print(np.setxor1d(sortiment_kb, sortiment_we))

['Chai' 'Crema' 'Filter']
['Chai' 'Crema' 'Decaf' 'Espresso' 'Filter' 'Kakao' 'Matcha']
['Decaf' 'Espresso']
['Decaf' 'Espresso' 'Kakao' 'Matcha']


# D – Bedingt ersetzen: `np.where`

## 8 – `np.where` als „Wenn-Dann-Sonst" für ganze Arrays

```python
np.where(bedingung, wert_wenn_True, wert_wenn_False)
```

Für **jedes Element** wird geprüft: Ist die Bedingung erfüllt, kommt der erste Wert ins Ergebnis, sonst der zweite.
Das Ergebnis ist ein **neues Array** mit der gleichen Form, das Original bleibt unverändert.

Vergleich: Beim Boolean Indexing (Notebook 1) **filtern** wir, das Ergebnis wird kürzer.
Bei `np.where` **ersetzen** wir, das Ergebnis bleibt gleich lang.

In [14]:
umsatz = np.array([820, 1350, 640, 1100, 980, 1520, 450])

kategorie = np.where(umsatz >= 1000, "hoch", "normal")
print(umsatz)
print(kategorie)

[ 820 1350  640 1100  980 1520  450]
['normal' 'hoch' 'normal' 'hoch' 'normal' 'hoch' 'normal']


Die Werte können auch **Arrays** sein, zum Beispiel für einen Bonus nur an starken Tagen:

In [15]:
# an Tagen ab 1000 € gibt es 5 % Bonus, sonst bleibt der Wert gleich
mit_bonus = np.where(umsatz >= 1000, umsatz * 1.05, umsatz)
print(mit_bonus)

[ 820.  1417.5  640.  1155.   980.  1596.   450. ]


## 9 – Mehr als zwei Fälle: `np.where` verschachteln

Für drei Kategorien setzt man ein zweites `np.where` an die Stelle von „sonst":

In [ ]:
umsatz = np.array([820, 1350, 640, 1100, 980, 1520, 450])

stufe = np.where(umsatz >= 1200, "sehr gut",
         np.where(umsatz >= 800, "ok", "schwach"))
print(stufe)

['ok' 'sehr gut' 'schwach' 'ok' 'ok' 'sehr gut' 'schwach']


## 10 – `np.where` mit Mengen kombinieren

`np.isin` liefert eine Maske, also eine Bedingung. Die passt direkt in `np.where`:

In [ ]:
stammkunden = np.array([101, 103, 110])
typ = np.where(np.isin(bestellungen_kb, stammkunden), "Stammkunde", "Laufkunde")
print(bestellungen_kb)
print(typ)

[101 102 103 101 104 102 101 105]
['Stammkunde' 'Laufkunde' 'Stammkunde' 'Stammkunde' 'Laufkunde'
 'Laufkunde' 'Stammkunde' 'Laufkunde']


> 💡 **Kurz erwähnt:** `np.where(bedingung)` **ohne** die zwei Werte gibt die **Positionen** zurück, an denen die Bedingung stimmt.
> Das ist nützlich, aber ein anderer Einsatz. Wir nutzen `np.where` mit drei Argumenten.

Lieferdauer in Tagen für acht Lieferungen:

In [18]:
liefertage = np.array([1, 3, 2, 5, 1, 4, 2, 6])

### Übung 4 – Lieferstatus

👥 **Gemeinsame Übung**

Nutze `liefertage` aus der Zelle oben.

1. Erstelle ein Array, das bei **bis zu 2 Tagen** `"pünktlich"` und sonst `"verspätet"` enthält.
2. Erstelle ein Array mit drei Stufen: bis 2 Tage `"pünktlich"`, bis 4 Tage `"knapp"`, darüber `"verspätet"`.
3. Für jede verspätete Lieferung (mehr als 2 Tage) gibt es **10 € Gutschrift**, sonst 0 €. Erstelle das Gutschrift-Array.

In [45]:
liefertage = np.array([1, 3, 2, 5, 1, 4, 2, 6])
# Dein Code hier
# 1 
print(np.where(liefertage <= 2, "pünktlich", "verspätet"))
# 2
print(np.where(liefertage <= 2, "pünktlich", 
      np.where(liefertage <= 4, "knapp", "verspätet")))
# 3
gutschrift = np.where(liefertage > 2, 10, 0)
print(gutschrift)


['pünktlich' 'verspätet' 'pünktlich' 'verspätet' 'pünktlich' 'verspätet'
 'pünktlich' 'verspätet']
['pünktlich' 'knapp' 'pünktlich' 'verspätet' 'pünktlich' 'knapp'
 'pünktlich' 'verspätet']
[ 0 10  0 10  0 10  0 10]


# E – Fancy Indexing: mehrere Positionen auf einmal

## 11 – Mit einer Liste von Positionen auswählen

Beim Slicing bekommt man nur **zusammenhängende** Bereiche (oder regelmäßige Abstände).
Mit **Fancy Indexing** gibt man eine **Liste von Positionen** an und bekommt genau diese Elemente, in dieser Reihenfolge:

```python
arr[[0, 3, 5]]      # doppelte Klammer: außen Indexing, innen die Liste
```

In [48]:
umsatz = np.array([820, 1350, 640, 1100, 980, 1520, 450])
#                    0          2          4                
tage = np.array(["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"])
# print(umsatz)

print(umsatz[[0, 2, 4]])        # Mo, Mi, Fr
print(tage[[6, 5]])            # So, Sa (Reihenfolge wie angegeben)

[820 640 980]
['So' 'Sa']


## 12 – Werte zuweisen

Wie beim Boolean Indexing kann Fancy Indexing links vom `=` stehen:

In [ ]:
umsatz = np.array([820, 1350, 640, 1100, 980, 1520, 450])
korrigiert = umsatz.copy()
korrigiert[[2, 6]] = 0          # Mi und So waren Inventurtage
print(korrigiert)

[ 820 1350    0 1100  980 1520    0]


## 13 – Bei 2D-Arrays: Spalten oder Zeilen auswählen

Fancy Indexing lässt sich mit `:` kombinieren. So wählst du bestimmte **Spalten** oder **Zeilen** aus:

In [49]:
filialen = np.array(["Kreuzberg", "Neukölln", "Friedrichshain", "Wedding"])
woche = np.array([
    [820, 950, 780, 1100, 1340, 1500, 1200],
    [610, 700, 690,  720,  980, 1150,  900],
    [900, 870, 910,  950, 1210, 1380, 1250],
    [540, 600, 580,  610,  800,  950,  700],
])

print(woche[:, [0, 4]])      # alle Filialen, nur Mo und Fr
print()
print(woche[[0, 2], :])      # nur Kreuzberg und Friedrichshain

[[ 820 1340]
 [ 610  980]
 [ 900 1210]
 [ 540  800]]

[[ 820  950  780 1100 1340 1500 1200]
 [ 900  870  910  950 1210 1380 1250]]


> 💡 **Kurz erwähnt – Kopie oder View?**
> Slicing liefert eine **View** (Fenster auf das Original, siehe Notebook 1).
> Fancy Indexing und Boolean Indexing liefern dagegen immer eine **Kopie**. Änderungen daran wirken nicht auf das Original zurück.

In [23]:
auswahl = umsatz[[0, 1]]    # Fancy Indexing -> Kopie
auswahl[0] = 0
print("Original:", umsatz)  # unverändert

fenster = umsatz[0:2]       # Slicing -> View
fenster[0] = 0
print("Original:", umsatz)  # geändert!

Original: [ 820 1350  640 1100  980 1520  450]
Original: [   0 1350  640 1100  980 1520  450]


> 💡 **Kurz erwähnt:** Man kann auch in **beiden** Dimensionen gleichzeitig Listen angeben, z. B. `woche[[0, 2], [1, 3]]`.
> Das Ergebnis ist dann aber **nicht** der Block, den man erwartet, sondern einzelne Paare (Zeile 0/Spalte 1 und Zeile 2/Spalte 3).
> Das brauchen wir im Kurs nicht.

### Übung 5 – Ausgewählte Tage

👥 **Gemeinsame Übung**

Arbeite mit `woche` (4 Filialen × 7 Tage, Mo–So) und `filialen`.

1. Gib die Umsätze **aller Filialen** am **Dienstag und Donnerstag** aus.
2. Gib die **Namen** von Filiale 3 und Filiale 1 aus (in dieser Reihenfolge).
3. Gib die **Wochenendumsätze** (Sa, So) von **Neukölln und Wedding** aus.

In [50]:
# Dein Code hier
woche = np.array([
    [820, 950, 780, 1100, 1340, 1500, 1200],
    [610, 700, 690,  720,  980, 1150,  900],
    [900, 870, 910,  950, 1210, 1380, 1250],
    [540, 600, 580,  610,  800,  950,  700],
])
filialen = np.array(["Kreuzberg", "Neukölln", "Friedrichshain", "Wedding"])
# 1 Di=1, Do=3
print(woche[:, [1, 3]])
# 2
print(filialen[[2, 0]])
# 3
print(woche[[1,3], 5:])
 

[[ 950 1100]
 [ 700  720]
 [ 870  950]
 [ 600  610]]
['Friedrichshain' 'Kreuzberg']
[[1150  900]
 [ 950  700]]


# F – Übungen zum selbstständigen Arbeiten

Hier übst du allein oder zu zweit. Die Übungen sind nach **Themen** sortiert.
Bei jeder Übung steht, zu welchem Block sie gehört. Wenn du bei einer Aufgabe hängst, schau im passenden Block nach.

## 14 – Daten für die Übungen

Führe diese Zelle zuerst aus. Sie erzeugt die Bestellungen eines ganzen Monats:

- `best_kb`, `best_nk`: Kunden-IDs aller Bestellungen in Kreuzberg und Neukölln (eine ID pro Bestellung)
- `best_wert`: Bestellwert in € zu jeder Bestellung in Kreuzberg (gleiche Reihenfolge wie `best_kb`)
- `newsletter`: Kunden-IDs, die den Newsletter abonniert haben

In [25]:
rng = np.random.default_rng(11)

best_kb = rng.integers(100, 140, size=40)
best_nk = rng.integers(120, 160, size=35)
best_wert = rng.integers(4, 40, size=40)
newsletter = np.array([101, 105, 112, 120, 125, 133, 138, 150, 155])

print("Kreuzberg:", best_kb)
print("Neukölln :", best_nk)
print("Werte KB :", best_wert)

Kreuzberg: [105 105 131 119 123 124 128 101 119 105 116 137 121 102 121 105 130 137
 139 124 134 114 105 120 117 126 139 111 134 105 113 131 109 126 118 120
 137 132 133 121]
Neukölln : [159 159 125 128 132 142 152 139 159 134 157 143 148 129 143 152 155 154
 158 125 151 138 147 131 120 123 158 155 132 137 129 125 154 146 123]
Werte KB : [11 24 36 39 11 26  5 10 11 19 16 30 20 16 36 26 29 30 16 36  4 15  9  4
 39  7 20 33 28 34  5 21  5 10 34  6 25  4 15 37]


## 15 – Thema: Mengen mit `set`

### Übung 6 – Newsletter-Aktion

📌 **Thema:** Mengen mit `set` (Block A)

Wandle die Arrays mit `set(arr.tolist())` in Python-Mengen um und beantworte:

(`.tolist()` macht aus dem Array eine normale Python-Liste. Ohne das zeigt Python die Zahlen als `np.int64(...)` an.)

1. Welche Newsletter-Abonnenten haben in **Kreuzberg** bestellt?
2. Welche Newsletter-Abonnenten haben in **keiner** der beiden Filialen bestellt?
3. Sind alle Newsletter-Abonnenten auch Kunden? (Tipp: Teilmenge ⊆ heißt in Python `<=`)

In [26]:
# Dein Code hier

## 16 – Thema: `np.unique` und `np.isin`

### Übung 7 – Stammkunden finden

📌 **Thema:** `np.unique` und `np.isin` (Block B)

Arbeite mit `best_kb`.

1. Wie viele Bestellungen und wie viele **verschiedene Kunden** hatte Kreuzberg im Monat?
2. Welche Kunden haben **mindestens 3-mal** bestellt?
3. Wie viel Prozent der Kunden haben nur **ein einziges Mal** bestellt?

In [27]:
# Dein Code hier

### Übung 8 – Umsatz der Newsletter-Kunden

📌 **Thema:** `np.unique` und `np.isin` (Block B)

Arbeite mit `best_kb`, `best_wert` und `newsletter`.

1. Erstelle eine Maske: Welche Bestellungen kamen von Newsletter-Abonnenten?
2. Gib die **Bestellwerte** dieser Bestellungen aus.
3. Gib die Bestellwerte aller **anderen** Bestellungen aus.

In [28]:
# Dein Code hier

## 17 – Thema: Mengenoperationen

### Übung 9 – Filialvergleich

📌 **Thema:** Mengenoperationen (Block C)

Arbeite mit `best_kb` und `best_nk`.

1. Welche Kunden haben in **beiden** Filialen bestellt? Wie viele sind das?
2. Wie viele **verschiedene Kunden** hat Kiez Kaffee in diesen beiden Filialen insgesamt?
3. Welche Kunden waren **nur in Neukölln**?
4. Prüfe die Formel aus der Mathe: |A ∪ B| = |A| + |B| − |A ∩ B|

In [29]:
# Dein Code hier

### Übung 10 – Zielgruppen für eine Aktion

📌 **Thema:** Mengenoperationen (Block C)

Kiez Kaffee plant zwei Aktionen:

1. **Willkommensgutschein:** für alle Newsletter-Abonnenten, die **noch nie** bestellt haben (in keiner der beiden Filialen).
2. **Filial-Tipp:** für alle Kunden, die in **genau einer** der beiden Filialen bestellt haben.

Berechne beide Zielgruppen mit NumPy-Funktionen.

In [30]:
# Dein Code hier

## 18 – Thema: `np.where`

### Übung 11 – Bestellungen einstufen

📌 **Thema:** `np.where` (Block D)

Arbeite mit `best_wert`.

1. Stufe jede Bestellung ein: ab 25 € `"groß"`, sonst `"klein"`.
2. Bei Bestellungen **ab 30 €** gibt es **10 % Rabatt**. Berechne die Endpreise (andere bleiben gleich).
3. Drei Stufen: unter 10 € `"S"`, unter 25 € `"M"`, sonst `"L"`.

In [31]:
# Dein Code hier

### Übung 12 – Kundenlabel

📌 **Thema:** `np.where` (Block D)

Erstelle für jede Bestellung in `best_kb` ein Label:

- `"Newsletter"`, wenn der Kunde den Newsletter hat,
- sonst `"beide Filialen"`, wenn der Kunde auch in Neukölln bestellt hat,
- sonst `"nur KB"`.

(Tipp: zwei Masken mit `np.isin`, dann `np.where` verschachteln.)

In [32]:
# Dein Code hier

## 19 – Thema: Fancy Indexing

### Übung 13 – Stichprobe prüfen

📌 **Thema:** Fancy Indexing (Block E)

Die Buchhaltung möchte die Bestellungen an den Positionen **0, 5, 10, 15, 20** prüfen.

1. Gib die Kunden-IDs **und** die Bestellwerte dieser Positionen aus.
2. Bei den Positionen **5 und 20** war der Wert falsch erfasst: Er muss jeweils **15 €** betragen.
   Korrigiere das in einer **Kopie** von `best_wert`.
3. Erzeuge die Positionsliste aus Aufgabe 1 mit `np.arange` statt von Hand.

In [33]:
# Dein Code hier

---
**Morgen:** Potenzen, Wurzeln und Logarithmen in NumPy: Rechnen mit ganzen Arrays und die Frage, welcher Datentyp wofür passt.